In [ ]:
''' Main Required Dependencies ''';
import xarray as xr
import numpy as np
import os # For use in verifying existing file paths (in the check_missing_individual_doy_datasets_stored function).

''' Optional Dependencies ''';
#from datetime import datetime                    # For a ~30 year period check function (Optional)
#from dateutil.relativedelta import relativedelta # For a ~30 year period check function (Optional)
#from dask.diagnostics import ProgressBar # To download processed datasets with a progress bar (Optional but HIGHLY recommended!)

''' Additional Optional Memory Monitoring Dependencies (memory monitoring is optional but INCREDIBLY ESPECIALLY recommended) ''';
#import psutil 
#import threading
#import time

''' Additional Optional Dependencies for Figure Creation ''';
import matplotlib.pyplot as plt # Required for the optional show_map (of the loaded data) function.
import cartopy.crs as ccrs # Required for the optional show_map (of the loaded data) function; provides map projection.

''' Additional Optional Dependencies for the File Validation, Verification, and Animation Section ''';
import glob # For loading many datasets at once (required to produce an animation)
import matplotlib.animation as animation # Required to produce an animation
import cartopy.feature as cfeature # To add land features and coastlines (Optional if you use something else)

<div style="background-color: #FFE099; padding: 10px; border: 3px solid #FFC233; text-align: center; font-family: Georgia, serif; font-weight: bold; white-space: pre;">||| - - - - - - - - - - - - - - - - - - - - - - - <|       SCRIPT NOTES       |> - - - - - - - - - - - - - - - - - - - - - - - |||</div>

<div style="background-color: #EFFAFA; border: 2px solid #A2E2E2; font-family: Georgia, serif; padding: 10px">
    <div style="text-align: center;">
        This is the script to <strong>process percentile thresholds</strong> and <strong>climatological means</strong> from your <strong>temperature data at various uniform depth levels.</strong>
    </div>
    <br>&#10148;&#xFE0E; This code calculates climatological means and percentile thresholds for 366 unique days of the year (including February 29) by default, using all available data.
    <br>&#10148;&#xFE0E; Default constants and settings are provided to show an example of the usage of this code to (ultimately) detect marine heatwaves based closely on Hobday et al. (2016).
    <br>&#10148;&#xFE0E; A fixed 30-year historical baseline period is used to calculate climatological means and percentiles by default. Adjust as need be (in the script-wide constants section).  
    <br>&#10148;&#xFE0E; You can duplicate this script and run its copies simultaneously to download more percentiles/means at once (if need be).
    <br>&#10148;&#xFE0E; If unusual or unexpected errors appear after attempting to run a cell again, restart the kernel!
    <br>&#10148;&#xFE0E; If you save many files, you may compare file sizes to see if any should be removed and redownloaded. Partially saved files will often have drastically different file sizes.
    <br>&#10148;&#xFE0E; For the default raw data loading code to function as intended, all subfolders' raw data files MUST share the same initial/ending naming convention.
    <br>&#10148;&#xFE0E; Lastly, in this script, constants define your root directory and output directories; you may tweak these and any constants 
    (and code) as you see fit to suit your needs!
    <br><br>
</div>

Things to be added in the next script version:
- A "full" region example (for calculating climatological means from); you would just need to have one subfolder 
  within the parent folder that you access.
- Validation against marineHeatWaves.py.
- Configuring the main percentile/mean calculating function to allow doy intervals or individual doy downloading 
  REGARDLESS of whether you are calculating/saving a percentile threshold or climatological mean (dataset). Currently, 
  the code is set to calculate and store datasets containing the data of a single day of the year for percentile 
  thresholds and more than one day of the year for climatological means.
- Remove any additional script redundancies or make tweaks for ease of use.
- Tweaking the animation function code to include a custom identifier argument.

<div style="color:#642CA9; padding: 10px; text-align: center; font-family: Georgia, serif; font-weight: bold; white-space: pre;">°º¤ø,¸¸,ø¤º°`°º¤ø,¸,ø¤°º¤ø,¸¸,ø¤º°`°º¤ø,¸    IMPORTANT SCRIPT-WIDE CONSTANTS AND FUNCTIONS    °º¤ø,¸¸,ø¤º°`°º¤ø,¸,ø¤°º¤ø,¸¸,ø¤º°`°º¤ø,¸
</div>

In [ ]:
''' - - - Constants set up - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - ''';
percentile = 90             # Desired (temperature) percentile threshold. 90 here indicates the 90th percentile. 
minutes_per_mem_update = 10 # For memory monitoring (optional). Minutes (roughly) per memory update (to keep track of its use and avoid crashing/issues).
memory_monitoring = True    # Required to be either False or True. Decide whether you want to use the optional memory monitoring feature (True) or not.

''' - - - Optional constants (if you want to run all cells) - - - - - - - - - - - - - - - - - - - - ''';
do_30yr_check = False   # Either False or True. If false, does not run the optional 30-year period check for your baseline slice below.
make_maps_of_loaded_data = True # Either False or True. If True, runs the show_map function, which shows loaded temperature data on a map.

''' - - - Historical baseline period setup - - - - - - - - - - - - - - - - - - - - - - - - - - - - ''';
## Pick a FIXED time period whose data you will use to calculate the climatological means/percentiles
baseline_period_slice_choice = slice('1993-01-01', '2022-12-31') # Included time period for mean/percentile calculations
baseline_folder_name = "Baseline9322" # Custom identifier (for you to set based on the chosen baseline period) for your saved means/percentiles

''' - - - Percentile/mean dataset filename/file path setup - - - - - - - - - - - - - - - - - - - - '''; 
# Instead of saving datasets with "clim" or "thresh", you may change these data type identifiers here.
climatological_means_id = "clim"
percentile_threshold_id = "thresh"

# These are the customizable filename variables; adjust as needed. 
custom_id_chosen = "300m_subset"  # A custom identifier to further distinguish the saved data with. You can leave this empty as "".
# Default output filename example: Atlantic_Top_thetao_thresh_90th_perc_300m_subset_265_Baseline9322.zarr; here, "300m_subset" was the custom_id_chosen.
# This example path above follows the format: 
# {*parent folder}_{*subfolder}_{*variable name}_{percentile_threshold_id}_{*default percentile info}_
# {custom_id_chosen}_{*day of the year}_{baseline_folder_name}.zarr
# All the * arguments are added later within the code or via inputs/constants that are not explicitly included here.

''' - - - Root and output directory setup - - - - - - - - - - - - - - - - - - - - - - - - - - - - - ''';
# These are the default directory conventions used here:
# Root directory: {my_root_directory}/
# Raw data directory: {my_root_directory}/{raw_data_folder_name}
# Climatological mean data directory: {my_root_directory}/{clim_data_folder_name}
# Percentile threshold data directory: {my_root_directory}/{perc_data_folder_name}
# In other words, within your root directory, you will have your raw data, climatological mean, and chosen percentile threshold dataset folders.

# Additionally, in the climatological mean and percentile threshold directories, a folder will be created named after your baseline_folder_name variable. 
# All the dataset outputs from this script will be stored in these baseline_folder_name folders. You can tweak this setup and the output directories.
# Final output path example:  
# d2/Thresh90th/Baseline9322/Atlantic/Atlantic_Top/Atlantic_Top_thetao_thresh_90th_perc_300m_subset_1_Baseline9322.zarr

my_root_directory = "/d2" # Should be your root directory, from which you access your data from and will save data to.
raw_data_folder_name = "Data"
clim_data_folder_name = "Clim"
perc_data_folder_name = f"Thresh{percentile}th"
raw_data_directory = f"{my_root_directory}/{raw_data_folder_name}"
clim_data_directory = f"{my_root_directory}/{clim_data_folder_name}"
perc_data_directory = f"{my_root_directory}/{perc_data_folder_name}"

''' - - - Print (verification) statements - - - - - - - - - - - - - - - - - - - - - - - - - - - - - ''';
print("CHOSEN CONSTANTS:\n")
print(f"Percentile threshold:                {percentile}th percentile")
print(f"Baseline slice object:               {baseline_period_slice_choice}")
print(f"Corresponding baseline identifier:   {baseline_folder_name}")
print(f"Memory monitoring:                   {memory_monitoring}\n")

print(f"Chosen root directory:                  {my_root_directory}")
print(f"Chosen raw data directory:              {raw_data_directory}")
print(f"Chosen output directory (means):        {clim_data_directory}")
print(f"Chosen output directory (percentiles):  {perc_data_directory}\n")

In [ ]:
''' - - - REQUIRED FUNCTIONS - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - ''';

''' ------------------------------------------------------------------------------------------------ '''
''' Function to normalize the unique day of the year value of each observed day in the format: 1-366 '''
''' ------------------------------------------------------------------------------------------------ '''

def normalize_dayofyear(time_coord):
    doy = time_coord.dt.dayofyear
    is_leap = time_coord.dt.is_leap_year

    # This code ensures March 1 is always the unique day of the year (doy) 61, regardless of leap year status.
    normalized_doy = xr.where(
        (~is_leap) & (doy >= 60),  # For a non-leap year, doy 60 is March 1 initially. As such, we shift forward
        doy + 1,                   # March 1 and any later days by 1 day. This makes March 1 have a doy of 61.
        doy                        # Otherwise, we keep the day of the year values as is for leap years and the Jan. to Feb. 28 period.
    )                              # Non-leap years end up having a missing doy 60 day by design.
    return normalized_doy


''' -------------------------------------------------------------------------------------------------------------------- '''
''' This function checks only STORED mean/percentile datasets that contain data for ONE unique day of the year (doy)     '''
''' using the default file storage settings. The doy values of any missing datasets (in the 1 to 366 calendar day range) ''' 
''' are returned in a list to be used by the main function that calculates percentiles/means.                            '''
''' -------------------------------------------------------------------------------------------------------------------- '''

def check_missing_individual_doy_datasets_stored(folder_name_arg, sub_folder_name_arg, baseline_name_arg, 
                                                 temp_data_var, custom_id, doy_start_arg, doy_end_arg, 
                                                 current_percentile=None, return_check_all=True):
    # This code can be adapted to whatever non-default file naming format you choose, assuming you keep note of 
    # which individual doy's data is contained in the accessed dataset(s) name (to access the doy number via a for loop).
    '''
    FUNCTION ARGUMENTS AND THEIR EXPLANATIONS

    folder_name_arg:     The name of the parent (region) folder you are accessing. 

    sub_folder_name_arg: The name of the (spatial subset) subfolder you are accessing.

    baseline_name_arg:   The baseline identifier indicating which fixed baseline period's data was used to calculate the
                         mean/percentile datasets you intend to access (that is also included in their filenames by default).
                         
    temp_data_var:       Name of the temperature variable in your dataset that you are accessing and calculating the 
                         means/percentiles for; it is included in the filename of the saved datasets.

    custom_id:           A custom identifier included in all the mean/percentile datasets' filenames, such as 
                         "{max_depth}m_subset" in the examples used throughout this script. This can be blank if applicable.

    doy_start_arg:       Initial day of the year (doy) value in the interval to be checked for missingness (of individual
                         single-doy mean/percentile datasets).

    doy_end_arg:         Final day of the year (doy) value in the interval to be checked for missingness (of individual
                         single-doy mean/percentile datasets). The code checks if all the doy mean/percentile datasets 
                         from doy_start_arg to doy_end_arg are present in the (default) directory they are expected to be in.

    current_percentile:  The integer percentile value of the current percentile threshold dataset (if applicable); if a 
                         climatological means dataset is being accessed instead, current_percentile should be set to None.
                         This helps with identifying the correct file paths to check (for missingness).

    return_check_all:    If set to True, this will allow a print statement of the total number of missing doy datasets to run.
    '''

    # Initialize a counter and list to update based on the number of missing datasets
    missing_counter = 0
    missing_list = []

    # Adjust the custom_id for the filename
    custom_file_id = f"{custom_id}_" if custom_id != "" else ""

    # Determine what type of file (means/percentiles) to read from its respective (default) directory
    calculating_means = True if current_percentile == None else False
    file_type_id = climatological_means_id if calculating_means else percentile_threshold_id
    file_type_directory = clim_data_directory if calculating_means else perc_data_directory
    perc_info = "" if calculating_means else f"{current_percentile}th_perc_" 

    # Check for the presence of INDIVIDUAL datasets within the 1 to 366 range for the full calendar year (by default)
    for i in range(1, 366+1): 
        # We check ONLY within the desired doy interval
        if i >= doy_start_arg and i <= doy_end_arg:

            # Do ensure these file paths are correct for your set up if you have changed any of the default directory settings
            id_path = f"{folder_name_arg}_{sub_folder_name_arg}"
            # Output example: Atlantic_Top

            filename = f"{id_path}_{temp_data_var}_{file_type_id}_{perc_info}{custom_file_id}{i}_{baseline_name_arg}.zarr"
            # Output (percentile dataset) example: 
            # Atlantic_Top_thetao_thresh_90th_perc_300m_subset_254_Baseline9322.zarr

            filepath = f'{file_type_directory}/{baseline_name_arg}/{folder_name_arg}/{id_path}/{filename}'
            # Output example: 
            # d2/Thresh90th/Baseline9322/Atlantic/Atlantic_Top/Atlantic_Top_thetao_thresh_90th_perc_300m_subset_254_Baseline9322.zarr

            # Now, we check if the filepath exists in the target location; if it does not, we add it to the missing_list!
            if not os.path.exists(filepath):
                missing_counter += 1
                missing_list.append(i)

    # If there are no missing doy datasets in our storage, we return None; otherwise, we return the list of missing doys!
    if missing_counter == 0:
        print("All doys checked and present in storage! (You should also check file sizes to verify everything downloaded correctly!")
        return None
    else:
        if return_check_all:
            print(f"All doys checked; you are missing {missing_counter} doy datasets in total!\n")

        return missing_list


''' -------------------------------------------------------------------------------------- '''
''' Function for finding already stored files for the interpolation of the Feb. 29 dataset '''
''' -------------------------------------------------------------------------------------- '''

# Function for finding the files necessary to interpolate for Feb 29
def find_file_and_return_it(folder_name_input, sub_folder_name_input, baseline_name_input,
                            temp_data_var_input, custom_id_input, percentile_input, 
                            current_doy=None, starting_doy=None, ending_doy=None):
    
    '''
    This function's arguments are automatically inputted in the main function. However, it could be used
    to load datasets that follow the established, default directory system and file-naming conventions
    used in this script. These are the inputs' descriptions if you would like to load a dataset independently 
    of the provided code/functions:
    
    folder_name_input:     The name of the parent (region) folder you are accessing.
    
    sub_folder_name_input: The name of the (spatial subset) subfolder you are accessing.
    
    baseline_name_input:   The baseline identifier indicating which fixed baseline period's data was used to calculate the
                           mean/percentile datasets you intend to access (that is also included in their filenames by default).
                         
    temp_data_var_input:   Name of the temperature variable in your dataset that you are accessing and calculating the 
                           means/percentiles for; it is included in the filename of the saved datasets.
                         
    custom_id_input:       A custom identifier included in all the mean/percentile datasets' filenames, such as 
                           "{max_depth}m_subset" in the examples used throughout this script. This can be blank if applicable.
                         
    percentile_input:      The integer percentile value of the current percentile threshold dataset (if applicable); if a 
                           climatological means dataset is being accessed instead, current_percentile should be set to None.
                           This helps with identifying the correct file paths to check (for missingness).
                           
    current_doy:           This should be either None or an integer value, which represents the value of the unique day of the year (doy).
                           This accesses datasets that include a single doy value; must be either None OR an integer value while 
                           starting_doy and ending_doy are both set to None.
                           Example settings: (current_doy=12, starting_doy=None, ending_doy=None) in the function call.
                           
    starting_doy:          This should be either None or an integer value, which represents the first value of the unique days of the 
                           year (doy) in a dataset containing multiple doy values. When a value is provided for this parameter, a value must
                           also be provided for ending_doy and current_doy must be set to None. 
                           Example settings: (current_doy=None, starting_doy=1, ending_doy=20) in the function call.
                           
    ending_doy:            This should be either None or an integer value, which represents the final value of the unique days of the 
                           year (doy) in a dataset containing multiple doy values. When a value is provided for this parameter, a value must
                           also be provided for starting_doy and current_doy must be set to None. 
                           Example settings: (current_doy=None, starting_doy=1, ending_doy=20) in the function call.
    '''
    
    # Determine if loading a climatological means or percentile threshold dataset
    calc_means = True if percentile_input == None else False
    
    # Adjust the custom_id for the filename
    custom_path_id = f"{custom_id_input}_" if custom_id_input != "" else ""

    # Determine what type of file (means/percentiles) to read from its respective (default) directory
    ds_type_id = climatological_means_id if calc_means else percentile_threshold_id
    ds_type_directory = clim_data_directory if calc_means else perc_data_directory
    perc_info = "" if calc_means else f"{percentile_input}th_perc_"  # Default; can be tweaked or removed.

    # Bool that reflects if both values in the doy interval (for multiple doy dataset saving) were provided
    interval_provided = True if (starting_doy != None and ending_doy != None) else False
    
    # Doy values based on target multi or single-doy datasets to access (based on inputs)
    doy_values = f"{current_doy}" if not interval_provided else f"{starting_doy}_to_{ending_doy}"
    
    # Determine the id path (to use in the directory and file name, by default)
    id_path = f"{folder_name_input}_{sub_folder_name_input}"

    # Setting up the filename and filepath to access
    filename = f"{id_path}_{temp_data_var_input}_{ds_type_id}_{perc_info}{custom_path_id}{doy_values}_{baseline_name_input}.zarr"
    filepath = f'{ds_type_directory}/{baseline_name_input}/{folder_name_input}/{id_path}/{filename}'
    
    
    # We check if the path/file exists in the target location
    if os.path.exists(filepath):
        # Quick adjustment to the final dataset; squeezing (dropping) the normalized_doy dimension if the dataset contains a single doy
        if interval_provided:
            ds = xr.open_zarr(filepath) 
        else:
            ds = xr.open_zarr(filepath).squeeze('normalized_doy', drop=True)
            
        return True, ds
    else:
        return False, None
    
    
''' -------------------------------------------------------------------------------- '''
''' Function to save the percentile threshold/climatological mean dataset to storage '''
''' -------------------------------------------------------------------------------- '''

def save_dataset_to_storage(folder_name_arg, sub_folder_name_arg, baseline_name_arg,
                            temp_data_var, custom_id_arg, show_debug_arg, single_download_arg, 
                            current_percentile=None, current_doy=None, 
                            start_val_arg = None, end_val_arg = None, 
                            ds_to_save = None):
    '''
    FUNCTION ARGUMENTS AND THEIR EXPLANATIONS

    folder_name_arg:     The name of the parent (region) folder you are accessing. 

    sub_folder_name_arg: The name of the (spatial subset) subfolder you are accessing.

    baseline_name_arg:   The baseline identifier indicating which fixed baseline period's data was used to calculate the
                         mean/percentile datasets you intend to access (that is also included in their filenames by default).

    temp_data_var:       Name of the temperature variable in your dataset that you are accessing and calculating the 
                         means/percentiles for; it is included in the filename of the saved datasets.

    custom_id:           A custom identifier included in all the mean/percentile datasets' filenames, such as 
                         "300m_subset" in the examples used throughout this script.

    show_debug_arg:      This parameter is meant to receive the main function's show_debug argument value. If set to True, 
                         this prevents the final resulting dataset from being saved (it only shows the output dataset and 
                         this function's debug statements up to producing the final product).

    current_percentile:  The percentile threshold of the current percentile threshold dataset (if applicable); if a 
                         climatological means dataset is being accessed instead, current_percentile should be set to None.
                         This helps with identifying the correct file paths to use.

    current_doy:         For saving datasets that include a single doy value; must be either None OR set to a (doy) value (from the
                         main function that calculates percentiles/means) while start_val_arg and end_val_arg are both set to None.
                         Example settings: (current_doy=12, start_val_arg=None, end_val_arg=None) in the function call.

    start_val_arg:       For saving datasets that include more than one doy value; both start_val_arg and end_val_arg must be either 
                         set to None OR set to unique (doy) values (from the main function that calculates percentiles/means) while 
                         current_doy is set to None. This is the first day of the year (doy) value in the interval (of doys whose data
                         is included in the dataset) to be saved and incorporated in the saved percentile/mean dataset's filename.
                         Example settings: (current_doy=None, start_val_arg=1, end_val_arg=2) in the function call.

    end_val_arg:         For saving datasets that include more than one doy value; both start_val_arg and end_val_arg must be either 
                         set to None OR set to unique (doy) values (from the main function that calculates percentiles/means) while 
                         current_doy is set to None. This is the last day of the year (doy) value in the interval (of doys whose data
                         is included in the dataset) to be saved and incorporated in the saved percentile/mean dataset's filename.
                         Example settings: (current_doy=None, start_val_arg=1, end_val_arg=2) in the function call.



    ds_to_save:          The dataset to be saved with the filepath constructed in this function (based on the function's given arguments).
                         It can be further filtered/adjusted here before saving with additional modifications of this code (not included here).
    '''

    # We check if we are saving a climatology dataset or percentile one
    calc_means = True if current_percentile == None else False

    ## Running a few checks to ensure either individual doy datasets or datasets with multiple doys are being saved, not both at once.

    # Bool that reflects if both values in the doy interval (for multiple doy dataset saving) were provided
    interval_provided = True if (start_val_arg != None and end_val_arg != None) else False

    # Checking that correct arguments were provided (both doy interval values or just a single current_doy)
    if current_doy != None and end_val_arg != None:
        resolution_msg1 = "Please provide either a value for current_doy OR values for start_val_arg and end_val_arg"
        resolution_msg2 = "Leave the other choice's paramaters as None"
        raise ValueError(f"SAVE DATASET TO STORAGE FUNCTION ERROR: Conflicting doy arguments provided.\n{resolution_msg1}. {resolution_msg2}.")
    elif current_doy != None and end_val_arg != None:
        raise ValueError(f"SAVE DATASET TO STORAGE FUNCTION ERROR: Conflicting doy arguments provided.\n{resolution_msg1}. {resolution_msg2}.")
        
    interv_msg1 = "Please provide a value if saving a dataset with the means/percentiles of multiple doys"
    if start_val_arg != None and end_val_arg == None:
        raise ValueError(f"SAVE DATASET TO STORAGE FUNCTION ERROR: Missing end_val_arg value.\n{interv_msg1}.")
    if end_val_arg != None and start_val_arg == None:
        raise ValueError(f"SAVE DATASET TO STORAGE FUNCTION ERROR: Missing start_val_arg value.\n{interv_msg1}.")               

    if current_doy == None and start_val_arg == None and end_val_arg == None:
        all_missing_msg = "No doy values provided.\nProvide values for either: current_doy OR start_val_arg and end_val_arg"
        raise ValueError(f"SAVE DATASET TO STORAGE FUNCTION ERROR: {all_missing_msg}.")
        
    if interval_provided:
        if start_val_arg == end_val_arg:
            multi_fix_msg = "Please ensure the two inputted values are different (if saving a dataset with multiple days of the year)"
            raise ValueError(f"SAVE DATASET TO STORAGE FUNCTION ERROR: Same start_val_arg and end_val_arg values.\n{multi_fix_msg}.")
        
    # Checking if a single doy's data is going to be saved
    single_doy_data_saving = None # Used to check for any errors not caught by the prior checks

    if current_doy != None and not interval_provided: 
        single_doy_data_saving = True
    if interval_provided and current_doy == None:
        single_doy_data_saving = False
    if single_doy_data_saving == None:
        err_doy_msg = "Conflicting doy arguments provided; this case was not covered by an earlier check"
        raise ValueError(f"SAVE DATASET TO STORAGE FUNCTION ERROR: {err_doy_msg}. Please debug.")

    # Proceeding with individual doy dataset saving
    if single_doy_data_saving:
        final_ds_to_save = ds_to_save          
    # Proceeding with multiple doy dataset saving
    else:
        print("Starting doy for current subset: ", start_val_arg)
        print("Ending doy for current subset: ", end_val_arg, '\n')
        final_ds_to_save = ds_to_save.sel({'normalized_doy': slice(start_val_arg, end_val_arg)})               


    ## File path set up 

    # Adjust the custom_id for the filename
    custom_path_id = f"{custom_id_arg}_" if custom_id_arg != "" else ""

    # Determine what type of file (means/percentiles) to read from its respective (default) directory
    ds_type_id = climatological_means_id if calc_means else percentile_threshold_id
    ds_type_directory = clim_data_directory if calc_means else perc_data_directory
    perc_info = "" if calc_means else f"{current_percentile}th_perc_"  # Default; can be tweaked or removed.

    # Do ensure these file paths are correct for your set up if you have changed any of the default directory settings
    full_id_path = f"{folder_name_arg}_{sub_folder_name_arg}"
    # Output example: Atlantic_Top

    # Doy values based on multi or single-doy dataset saving
    doy_values = f"{current_doy}" if single_doy_data_saving else f"{start_val_arg}_to_{end_val_arg}"
    
    filename = f"{full_id_path}_{temp_data_var}_{ds_type_id}_{perc_info}{custom_path_id}{doy_values}_{baseline_name_arg}.zarr"
    # Output (percentile dataset) example: 
    # Atlantic_Top_thetao_thresh_90th_perc_300m_subset_254_Baseline9322.zarr

    filepath = f'{ds_type_directory}/{baseline_name_arg}/{folder_name_arg}/{full_id_path}/{filename}'

    # Specific informative print statement for climatological means
    if calc_means:
        print("Current climatology dataset to save:\n", final_ds_to_save, '\n')
    # Specific informative print statement for percentile thresholds  
    else:
        print(f"Current {current_percentile}th percentile dataset to save:\n", final_ds_to_save, '\n')
    print("Filepath of dataset: ", filepath)


    ## Continue to saving or stop (if we are at the end of the debug)
    if show_debug_arg:
        raise ValueError('End of debug. Proceed with the setting "show_debug = False" to start saving the percentile thresholds.')
    # Not debugging; proceed with saving!
    else:
        try: # This should run if the optional ProgressBar dependency was fulfilled.
            with ProgressBar():
                final_ds_to_save.to_zarr(filepath, mode='w', consolidated=True)
        except Exception as e:
            final_ds_to_save.to_zarr(filepath, mode='w', consolidated=True)

        # Quick informative print update
        if not single_doy_data_saving:
            print(f"Saved subset: doys {start_val_arg} to {end_val_arg}")
        else:
            print(f"Saved subset: doys {current_doy}")

        print(f"Moving on!", "\n")
        print("---------------------------------------------------------------------------------------------------------")

        # Optional: for single downloads
        if single_download_arg:
            stop_monitoring = True
            raise ValueError("Single dataset file-saving finished. Please enter a new desired chunk starting value to begin from.")
    
print("All required functions loaded successfully!")

In [ ]:
''' - - - OPTIONAL FUNCTION - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - ''';

''' ----------------------------------------------- '''
''' Function to keep track of and show memory usage '''
''' ----------------------------------------------- ''' 

if memory_monitoring:
    def monitor_memory(interval_minutes=5, log_file=None):
        interval = interval_minutes * 60  
        
        while not stop_monitoring:
            mem = psutil.Process(os.getpid()).memory_info().rss / (1024**3)  # in GB
            print(f" | Memory usage: {mem:.2f} GB | Memory: {psutil.virtual_memory().percent}% used | ")
            
            if log_file:
                with open(log_file, 'a') as f:
                    f.write(f"{time.strftime('%Y-%m-%d %H:%M:%S')}: {mem:.2f} GB\n")
            time.sleep(interval)
    print("Memory function loaded!")
    
else:
    if "monitor_memory" in globals():
        del monitor_memory
    
    print('You have chosen to not use memory monitoring. If this was a mistake, update the constant "memory_monitoring" to True!')

In [ ]:
''' - - - OPTIONAL FUNCTION - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - ''';

if do_30yr_check:
    ''' ------------------------------------------------------------------------------- '''
    ''' Function to check whether the inputted time period is at least roughly 30 years '''
    ''' ------------------------------------------------------------------------------- '''

    def rough_30_year_period_check(time_slice, tolerance=0.01):
        start_str = time_slice.start
        stop_str = time_slice.stop

        start_date = datetime.strptime(start_str, '%Y-%m-%d')
        end_date = datetime.strptime(stop_str, '%Y-%m-%d')

        delta = relativedelta(end_date, start_date)
        total_years = delta.years + delta.months/12 + delta.days/365.25

        return total_years, abs(total_years - 30) <= tolerance

    # Running a rough time check for the baseline provided
    total_time, is_time_slice_30_years = rough_30_year_period_check(baseline_period_slice_choice)

    if not is_time_slice_30_years:
        error_message = "Please check that your chosen baseline time slice covers a 30 year period."
        raise ValueError(f"{error_message}.\n            The chosen slice covers roughly {total_time} years.")
    else:
        period_checked = f"{baseline_period_slice_choice.start} to {baseline_period_slice_choice.stop}"
        check_pass_msg_1 = f"The chosen baseline period of {period_checked} has been identified as roughly covering a 30 year period." 
        print(f"{check_pass_msg_1} Do ensure this is the case separately.\n")

else:
    print("Proceeding without checking if the baseline period (slice object) covers (roughly) 30 years!")

<div style="color:#CD6600; padding: 10px; text-align: center; font-family: Georgia, serif; font-weight: bold; white-space: pre;">°º¤ø,¸¸,ø¤º°`°º¤ø,¸,ø¤°º¤ø,¸¸,ø¤º°`°º¤ø,¸    LOADING TEMPERATURE DATASETS    °º¤ø,¸¸,ø¤º°`°º¤ø,¸,ø¤°º¤ø,¸¸,ø¤º°`°º¤ø,¸
</div>

Here, rather than loading entire datasets of temperatures at various depth levels all at once (which can possess incredibly
large file sizes and consequently lead to crashes during computations due to extreme memory usage), data from one or more
folders that contain spatially subset raw temperature data (not performed in this script) is instead loaded into memory as 
single (or combined) datasets from which means and percentiles are then calculated. Small datasets may not have this issue and may 
thus be loaded in a manner similar to that employed in the sst_climatology_and_percentile_calculator_v1.ipynb script, if your 
system's memory allows. Overall, working with smaller, spatially subset raw temperature datasets helps avoid issues like crashes 
from the get-go and is highly recommended over loading entire datasets first and then trying to process climatological means/percentile 
thresholds for a specific region/area of interest. 

It is recommended that when first running the code from this cell, you adjust and modify it to be able to load all of the data
subsets for your region of interest as one complete dataset; you should then keep any filtering and latitudinal/longitudinal adjustments 
of the datasets and use them when calculating percentiles/means (for efficiency and to reduce any difficulties later when combining the output
datasets together as one). Here, you can verify that everything loaded well using the optional show_map function if you have run the cell
and have its dependencies installed (and active). Your datasets should all use the same longitude format (ranging from 0 to 360 or from 
-180 to 180 degrees) to avoid issues when combining.

Now, while the base raw data directory remains what it was set to in the script-wide constants section above, the 
spatially subset data was stored and is accessed using the following format:

{my_root_directory}/{raw_data_folder_name}/{region_id_folder_name}/{region_id_folder_name}\_{region_id_subfolder_name}

(or equivalently: {raw_data_directory}/{region_id_folder_name}/{region_id_folder_name}\_{region_id_subfolder_name})

Examples:  
/d2/Data/Atlantic/Atlantic\_Top  
/d2/Data/Atlantic/Atlantic\_Bottom  
/d2/Data/Atlantic/Atlantic\_Right

In the examples above, temperature data encompassing the full North Atlantic Ocean region was stored across the three following 
region_id_subfolder_name folders: Atlantic_Bottom, Atlantic_Right, and Atlantic_Top, which contain data corresponding to 
different and easily identifiable ("top," "bottom," "right") arbitrary subregions (of the North Atlantic). Observably, these folders 
are all considered subfolders within the parent region_id_folder_name folder, that being the "Atlantic" folder, which is itself 
stored in the set raw data directory, /d2/Data/. This format is similarly used when saving the resulting calculated climatological 
means and percentile thresholds, with their respective directories being:

Climatological Mean Output Directory:
{my_root_directory}/{clim_data_folder_name}/{baseline_folder_name}/{region_id_folder_name}/{region_id_folder_name}\_{region_id_subfolder_name}
Example: /d2/Clim/Baseline9322/Atlantic/Atlantic_Bottom

Percentile Threshold Output Directory:
{my_root_directory}/{perc_data_folder_name}/{baseline_folder_name}/{region_id_folder_name}/{region_id_folder_name}\_{region_id_subfolder_name}
Example: /d2/Thresh90th/Baseline9322/Atlantic/Atlantic_Bottom

At the end of the day, these directory settings were set up for my personal convience; of course, you may choose to tweak or personalize 
the way you access and load your saved raw temperature dataset(s) with temperatures at various uniform levels of depth (non-uniform depth 
levels have not been tested here and would likely require some modification of this script's code).

Lastly, a few final notes included in the GitHub repository's README.md which are ESPECIALLY applicable here with large temperature 
datasets: 
* Based on PERSONAL TESTING, saving LARGER, MORE spatially AND temporally "complete" MEAN datasets for a desired region is HIGHLY recommended, at least time-wise for you, that is. To elaborate, it is more efficient and faster to produce climatological mean datasets that both encompass your COMPLETE region of interest AND contain day of the year (doy) INTERVALS instead of individual days. 

* THE OPPOSITE IS TRUE when calculating/saving percentile threshold datasets; crashes can often occur from high memory usage during percentile computations when using large amounts of raw data. Limiting the amount of data that is processed at a time, particularly via spatial subsetting (or subsetting via depth levels, which is not employed here), as well as by ONLY saving the data of a SINGLE day of the year at a time helps avoid crashes. Additionally, saving percentile threshold datasets can take several hours PER DAY OF THE YEAR, mostly depending on how much raw temperature data was provided; larger datasets take much longer to process and save than smaller datasets, so subset!

In [ ]:
''' - - - OPTIONAL MAP FEATURE (for the raw data loading code in the next cell) - -  - - - - - - - - - - - - ''';

if make_maps_of_loaded_data:
    ''' -------------------------------------------------------------------------------------------------------------------- '''
    ''' This function shows a global-scale plot of your raw data after it is loaded into single datasets (in the next cell)! '''
    ''' Please ensure the required dependencies (listed at the top of this script) are installed and run!                    '''
    ''' -------------------------------------------------------------------------------------------------------------------- '''

    # Remember to check this function's listed dependencies at the top of the script!.
    def show_map(ds_input, date = "2003-08-22", chosen_depth=0.5):
        # The data and depths are arbitrary; select options that are true for your dataset.

        ds_map = ds_input.sel(time=date, depth=chosen_depth, method='nearest')

        # Map figure set up
        fig, ax = plt.subplots(figsize=(10, 6), 
                               subplot_kw={'projection': ccrs.Mercator()})

        im = ax.pcolormesh(ds_map.longitude, ds_map.latitude, ds_map,
                           transform=ccrs.PlateCarree(),
                           cmap='RdYlBu_r')

        # Additional optional features (can be removed/tweaked)
        ax.set_extent([0, 360, -30, 90], crs=ccrs.PlateCarree())
        ax.coastlines()
        ax.gridlines(draw_labels=True)

        # Colorbar features
        cbar = plt.colorbar(im, ax=ax, shrink=0.7) 
        cbar.set_label('Sea Surface Temperature (°C)', rotation=270, labelpad=15)

        # Title
        ax.set_title(f'Sea Surface Temperature - {date}', fontsize=14)

        plt.show()
        plt.close(fig)

    print("Optional map feature successfully loaded!\n")
    
else:
    print("Proceeding without the optional map feature (for data loaded with the code in the next cell)!")

In [ ]:
'''  ---  Dataset loading (from the set raw data directory)  -------------------------------------------------------------  ''';

'''     
A list of dictionaries is used to access the raw data. Moreover, each separate dictionary within the list follows the following format:

{"region_id_folder_name": ["region_id_subfolder_name_1", "region_id_subfolder_name_2", ...]} 
for however many custom subregion subfolders you have (see the example setup below these comments).

        As seen in the setup examples above and below, you can have more than one subfolder per parent (region_id_folder_name) folder. 
Recognizably, sometimes you may only want to process one subfolder's data at a time. If that is the case, you can comment out the other 
subfolders' names. You can further comment out entire dictionaries if you are interested in working on a different dictionary without 
removing the others keys within their dictionaries, as shown below. Do make sure to keep note of what your chosen subfolder name identifiers 
are and what they mean to you (and also try to avoid vague identifiers)!
''';

region_dict_list = [ # Here, only the entire Atlantic region dictionary and its subfolders are loaded.
    {"Atlantic": ["Top",      # An arbitrary subset of the North Atlantic; the "top" subregion.
              #     "Bottom",  # An arbitrary subset of the North Atlantic; the "bottom" subregion.
               #    "Right"    # An arbitrary subset of the North Atlantic; the "right" subregion (relative to the combined top and bottom subregions)
                 ]},   
   # {"Pacific": ["Right", 
   #              "Left"]},
   # {"Mid": [
        #"LowLat",             
    #    "Remaining"   
   # ]}
]


'''    For loops (employed below) are used to call multiple subfolders within parent folders (like the "Top" and "Bottom" 
subfolders in the "Atlantic" parent folder within the example setup above). Based on your setup, the data of many or a few parent 
folders or subfolders can be combined and have their climatological means and percentile thresholds calculated for them. 

        It is recommended for you to attempt to load the data of all of the subfolders in a parent folder together as one dataset 
(if they comprise a single "complete" region, like in the "Atlantic" example used here), as this would allow you to identify
any further adjustments you would like to make for INDIVIDUAL subfolders' data. You could then load the data within each subfolder
alone and calculate means/percentiles to avoid high memory usage (from not having other datasets loaded and used in memory).

        On another note, for the for loop structure to work as intended, all subfolders' raw data files MUST share the same 
initial/ending naming convention; in the example below, all the data files begin with "daily_data" and end with ".zarr" (which should 
be changed to whatever file type extensions/names your datasets share). Additionally, for simplicity, in the for loop below, the previous 
example format's region_id_folder_name corresponds to folder_name and the region_id_subfolder_name corresponds to sub_folder_name.
''';


## For loops for accessing and loading the data of one or multiple subfolders inside a "complete" region's parent folder within the raw directory

# Loop over all the enabled (not commented) main folder dictionaries in the dictionary list
for folder in region_dict_list:
    
    # Access the folder_name of the current dictionary and its list of corresponding subfolder names
    for folder_name, sub_folders in folder.items(): 
        # Informational print statement (can be removed)
        print(f"Current folder: {folder_name}\n") 
    
        observed_data_directory = "" # Initializing a new, empty directory filepath
        folder_name_datasets = [] # Initializing a list to store subfolder datasets
        
        # Access individual subfolders (via their names in the subfolder list)
        for sub_folder_name in sub_folders: 
            # Informational print statement (can be removed)
            print(f"Accessing {folder_name} folder's subfolder: {sub_folder_name}\n") 
             
            # The default subfolder filepath as in the earlier Notes example; you may tweak it as need be.
            observed_data_directory = f'{raw_data_directory}/{folder_name}/{folder_name}_{sub_folder_name}'
            
            # Collect all the applicable raw data datasets 
            raw_data_files = f'{observed_data_directory}/daily_data*.zarr' # Adjust to identify your saved files as need be. 
            #   The * allows us to later grab all the datasets that end with ".nc" and start with "sst" here.

            # Load the subfolder's raw temperature dataset(s) as one with xarray
            ds = xr.open_mfdataset(         
                raw_data_files,             # Glob pattern (the * grabs all datasets)
                parallel=True,              # Enable parallel file opening 
                combine='by_coords',        # Merge based on coordinate values
                engine='zarr')              # Specify the engine (may not work with the wrong engine based on your dataset's format)
       
            ''' ----  CUSTOM RAW DATASET ADJUSTMENTS!  ------------------------------------------------------------------------------------------- '''
            ''' The following raw temperature dataset adjustments showcase how you could go about with adjusting your own data within the default  '''
            ''' directory system established here. You do not need these adjustments; they should be removed. You may also replace the adjustments '''
            ''' shown with your own adjustments if you have any that you want to make!                                                             '''
            ''' ---------------------------------------------------------------------------------------------------------------------------------- ''';
        
            min_longitude = min(ds.longitude.values)
            min_latitude = min(ds.latitude.values)
            max_latitude = max(ds.latitude.values)
            max_longitude = max(ds.longitude.values)
                
            # Adjusting the Atlantic folder's data
            if folder_name == "Atlantic":
                # If statements ensure both types of adjustments (latitudinal/longitudinal) are performed if different conditions are met
                if (min_longitude == -101):
                    ds = ds.sel(longitude=slice(-101, -14.001))
                if (min_latitude < 0):
                    ds = ds.sel(latitude=slice(-0.75, max_latitude))
                    
            # Adjusting the Pacific folder's data
            elif folder_name == "Pacific":
                if (max_longitude > 259):
                    ds = ds.sel(longitude=slice(110.001, 258.999))
                if (min_latitude < -1.5):
                    ds = ds.sel(latitude=slice(-0.75, 90.001))
                
            # Adjusting the Mid folder's data
            elif folder_name == "Mid":
                if sub_folder_name == "LowLat":
                    ds = ds.sel(latitude=slice(-20.9, -0.75))
                    ds = ds.where(~((ds.longitude > 19.999) & (ds.longitude < 49.001)), drop=True)
                    
            '''  ----------------------------------------------------------------------------------------------------------------------------- ''';
            
            ## Make sure to perform any further desired filtering/cleaning here.
            
            # Now, add the adjusted subset's loaded dataset into the folder_name_datasets list and repeat for the rest of the subfolders!
            folder_name_datasets.append(ds)

            
        '''  ---  CUSTOM CHUNKING!  ---------------------------------------------------------------------------------------  ''';
        
        '''  Zarr chunks to be used (based on the combined "complete" regions) on the final loaded datasets are set up here. '''
        '''  Chunk dimension sizes should divide the FINAL (COMBINED) dataset's coodinate sizes exactly to avoid any issues. '''
        '''  While the best_chunks are determined here, they can also be included in the custom raw dataset adjustments      '''
        '''  section above instead (although you would need to make sure there are no issues).                               '''
        
        if folder_name == "Atlantic":
            best_chunks = {'depth': -1, 'time': 28, 'latitude': 218, 'longitude': 242} 
            # The final dataset dimensions are (time: 11688, depth: 28, latitude: 1090, longitude: 1452)
            # For latitude: 1090 / 218 = 5, and for longitude: 1452 / 242 = 6. For time, 28 was chosen since a small value
            # was needed to avoid high memory use; 8 (since 8 divides 11688 exactly) was deemed too small, so 28 was arbitrarily chosen.
            
        # Other examples
        elif folder_name == "Pacific":
            best_chunks = {'depth': -1, 'time': 28, 'latitude': 218, 'longitude': 229}

        elif folder_name == "Mid": # Here, only one of the subfolder's datasets are loaded at a time, so I always use the correct best_chunks
            if sub_folder_name == "Remaining":
                best_chunks = {'depth': -1, 'time': 28, 'latitude': 221, 'longitude': 361}
            elif sub_folder_name == "LowLat":
                best_chunks = {'depth': -1, 'time': 28, 'latitude': 121, 'longitude': 361}
        
        '''  -------------------------------------------------------------------------------------------------------------- ''';
        
        
        ## Combining all the subfolder datasets stored in the folder_name_datasets list!
        
        # If we have many datasets appended to the list, we combine them
        if len(folder_name_datasets) > 1: 
            folder_combined_ds = xr.combine_by_coords(folder_name_datasets, compat='no_conflicts')
            
        # If there is just one dataset appended to the list, we ignore the list and proceed with the single, already adjusted datset
        else:
            folder_combined_ds = ds

        # Chunk with the set best_chunks (you can load the dataset earlier to help you determine the best chunk values)
        folder_combined_ds = folder_combined_ds.chunk(best_chunks)

        # Saving this dataset (as a reference) globally; you can call upon it as a normal variable {folder_name}_obs with the folder_name specified.
        # An example of calling this object is shown in the main mean/percentile calculation code.
        globals()[f'{folder_name}_obs'] = folder_combined_ds
        print(f'Combined dataset (in the variable "{folder_name}_obs"):\n', globals()[f'{folder_name}_obs'], '\n')

        # Optional map figure creation
        if make_maps_of_loaded_data:
            # Informational print statement (can be removed)
            print("Creating a figure using the outputted loaded raw temperature dataset...\n")
            
            try:
                show_map(globals()[f'{folder_name}_obs'].thetao) # Ensure you are accessing the appropriate variable data
            
            except Exception as e:
                err_msg1 = "Could not make a figure of the successfully loaded data"
                print(f"ERROR: {err_msg1}. Please ensure the show_map function and its required dependencies are run.") 
        else:
            print("Data successfully loaded! No map of the loaded data was generated based on make_maps_of_loaded_data being set to False. ")
        
        print(' ------------------------------------------------------------------------------------------------------------------------------ \n')

<div style="color:#104E8B; padding: 10px; text-align: center; font-family: Georgia, serif; font-weight: bold; white-space: pre;">°º¤ø,¸¸,ø¤º°`°º¤ø,¸,ø¤°º¤ø,¸¸,ø¤º°`°º¤ø,¸    CALCULATING MEANS & PERCENTILES    °º¤ø,¸¸,ø¤º°`°º¤ø,¸,ø¤°º¤ø,¸¸,ø¤º°`°º¤ø,¸</div>

In [ ]:
''' ----------------------------------------------------------------------------------------------------------------------- '''
''' Function to calculate the percentile threshold/climatological means of a temperature dataset with multiple depth levels '''
''' ----------------------------------------------------------------------------------------------------------------------- '''

# Function to calculate the percentile threshold values for specific depths
def calculate_perc_or_clim_given_a_percentile(temp_data, baseline_slice, 
                                              folder_name, sub_folder_name, baseline_name, 
                                              temp_data_var_name, custom_id, 
                                              optimal_chunks, window_half_width=5, 
                                              minutes_per_memory_update=5, chosen_percentile=90, 
                                              start_doy=1, end_doy=366, doy_batch_size=None, max_doy_chunking_val=61,
                                              show_debug=True, single_download=False, 
                                              missing_list=False, finish_at_missing_list_end=True):     
    if show_debug:
        debug_msg_1 = "You have set show_debug to true; this will show how the percentiles/means are processed"
        debug_msg_2 = "based on your inputted arguments \nand provide a preview of the output"
        debug_msg_3 = "If you are satisfied with the output (and your arguments), compute and"
        debug_msg_4 = "save the calculated percentiles/means by setting\nshow_debug to false"
        debug_msg = f"{debug_msg_1} {debug_msg_2}.\n{debug_msg_3} {debug_msg_4}.\n"
        print(debug_msg)
    
        print("---------------------------------------------------------------------------------------------------------")
        print("Part 0: Running a few quick error checks for the provided arguments!")
        print("---------------------------------------------------------------------------------------------------------\n")
              
    # Checking start and end bounds
    if start_doy < 1 or end_doy > 366:
        raise ValueError("Please provide a start_doy that is ≥ 1 and an end_doy that is ≤ 366.")
        
    if end_doy < start_doy:
        raise ValueError("Please make sure your end_doy is greater than your start_doy; these are your dataset processing bounds.")
        
 
    # Establishing if we calculating percentiles or means 
    calculate_means = True if (chosen_percentile == None) else False
   
    # Checking if we are performing batch-saving (or saving everything all at once)
    if doy_batch_size == None: # We are saving everything at once, no need to batch save
        batch_saving = False
        doy_batch_size = end_doy - start_doy + 1 # Setting this to the output day of the year coordinate (size) value 
    else:
        batch_saving = True # We are not saving everything at once but want to do so in batches!
    
    
    if show_debug: 
        print("All clear!\n")
        print("---------------------------------------------------------------------------------------------------------")
        print("Part 1: Assign normalized unique day of the year (doy) values to the sliced observation dataset")
        print("---------------------------------------------------------------------------------------------------------\n")
    
    print(f"Chosen baseline slice: {baseline_slice}")
    print(f"Chosen window half-width: {window_half_width}")
    info_msg_1 = f"This means we use the data in the {window_half_width} days before and after each"
    info_msg_2 = "central day of the year (including the central day) when calculating the central day's"
    print(f"{info_msg_1} {info_msg_2} climatological means/percentile thresholds.\n")
    
                             
    if calculate_means:
        print(f"Calculating means in batches of (at most) {doy_batch_size} doys between {start_doy} and {end_doy}.\n")
    else:
        percentile_used = chosen_percentile/100
        print(f"Final percentile used (in calculations): {percentile_used} ({chosen_percentile}th percentile)", '\n')
        print(f"Calculating percentiles in batches of (at most) {doy_batch_size} doys between {start_doy} and {end_doy}.\n")
    
    # Doy values for specific dates (for later)
    feb28_doy = 59
    feb29_doy = 60
    mar1_doy = 61
    
    # We create a missing doys chunk list for percentiles only, if desired
    if not calculate_means:
        if missing_list:
            missing_list = check_missing_individual_doy_datasets_stored(folder_name_arg=folder_name, sub_folder_name_arg=sub_folder_name, 
                                                                        baseline_name_arg=baseline_name, temp_data_var=temp_data_var_name, 
                                                                        custom_id = custom_id, doy_start_arg=start_doy, doy_end_arg=end_doy,
                                                                        current_percentile=chosen_percentile, return_check_all=show_debug)
        
            ## Quick check to make sure the returned missing doy list provided is valid.
            if missing_list is not None:
                if type(missing_list) is list:
                    filtered_list = [i for i in missing_list if i >= start_doy and i <= end_doy]

                    print("Missing doys within the set doy interval were found!\n") 
                    print(f"We are processing all missing doys within the filtered missing_list:\n{filtered_list}\n\n")
            else:
                raise ValueError("No missing doys were found in the doy interval! Turn off missing_list to proceed anyway!")
    
    ## Extracting baseline period data
    temp_baseline = temp_data.sel(time=baseline_slice)
    if show_debug: print("Original Thetao Baseline Period Data: ", '\n', temp_baseline, '\n')
   
    # Assigning normalized doy values to the baseline period dataset
    temp_norm = temp_baseline.assign_coords(
        normalized_doy=('time', normalize_dayofyear(temp_baseline.time).data))
    if show_debug: print("Temperatures with Normalized Doy: ", '\n', temp_norm, '\n')
        
    '''
    # Totally optional debug option here: show ALL normalized day of the year (doy) values;
    # all years are in the 366-day format, with some missing day 60 (feb 29)
    with np.printoptions(threshold=np.inf):
        print(temp_norm.normalized_doy.values) 
    '''
    

    if show_debug: 
        print("---------------------------------------------------------------------------------------------------------")
        print("Part 2: Get the actual doy values of the baseline period data (should be 1 - 366)")
        print("---------------------------------------------------------------------------------------------------------\n")
    unique_doys = np.unique(temp_norm.normalized_doy.data)
    unique_doys = unique_doys[~np.isnan(unique_doys)]  # Remove any NaN values
    unique_doys = unique_doys.astype(int)  # Ensure integer day-of-year values
    if show_debug: print(f"Found {len(unique_doys)} unique day-of-year values!")
    if show_debug: print("Unique doys:", '\n', unique_doys, '\n')

         
    global stop_monitoring
    if show_debug: 
        choice_message = "climatological mean" if calculate_means else "percentile threshold"
        print("---------------------------------------------------------------------------------------------------------")
        print(f"Part 3: Calculate the desired {choice_message} data for the desired doy(s).")
        print("---------------------------------------------------------------------------------------------------------\n")
        stop_monitoring = True # We don't want to start showing memory use.
    else:
        # We start monitoring here so that it only runs once
        stop_monitoring = False # We do want to start showing memory use.
        monitor_thread = threading.Thread(target=monitor_memory, kwargs={'interval_minutes': minutes_per_memory_update})
        monitor_thread.daemon = True
        monitor_thread.start()
        
    # Error messages for later
    missing_list_message_finished = "All missing doys within the missing doy list (except Feb 29) processed!\n"
    missing_message_finished = "Set missing end reached!\n"
    no_feb29_possible_warning = "WARNING: Cannot interpolate Feb 29; missing Feb 28 or Mar 1 data!\n"
    
    # Initialize a dictionary for the climatological means if wanted
    if calculate_means:
        seas_clim_dict = {}
        
    # Bool for debug purposes
    shown_once = False
 
    # Loop for doys 1 - 366 (excluding Feb 29, doy 60)
    for doy in unique_doys:
        # We skip February 29th (to interpolate later)
        if doy == feb29_doy:  
            continue # Note: doy 60 data is still used within the appropriate window_data when available
        
        # These next checks only run for percentile calculations; all doys are loaded in the climatological dictionary (except Feb 29)
        if not calculate_means:
            # We skip doys to begin on the desired chunk_start value
            if doy < start_doy: 
                continue

            # Now we look at doys greater than the end_doy
            if doy > end_doy:
                # For percentiles, we check if we have a chunk list and whether we wish to continue past the set end_doy bound
                if missing_list:
                    if finish_at_missing_list_end: # If we want to stop at the end of the set chunk interval:
                        stop_monitoring = True
                        raise ValueError(missing_list_message_finished)

                # And run this code otherwise for doys beyond the set chunk interval
                else: 
                    if finish_at_missing_list_end:
                        stop_monitoring = True
                        raise ValueError(missing_message_finished)

            # for percentiles, we skip any doys that are not found in the list of missing doys created above (if the chunk_list was set to true)
            if missing_list:
                if doy not in missing_list:
                    continue 
        
        # Create window around this DOY
        window_doys = []
        
        for w in range(-window_half_width, window_half_width + 1):
            target_doy = doy + w
            
            if show_debug and not shown_once: 
                print("Day of the year: ", doy, "| Target Window Index: ", w, "| Target Window Value: ", target_doy)

            # Handle year wraparound properly
            if target_doy < 1:
                target_doy += 366
            elif target_doy > 366:
                target_doy -= 366
            
            # Handle year boundaries by keeping only valid doys
            if target_doy in unique_doys:
                window_doys.append(target_doy)
            
            if show_debug and not shown_once: print("Window Doys: ", window_doys, '\n')

        # Now, we select the data for this window
        window_data = temp_norm.where(temp_norm.normalized_doy.isin(window_doys), drop=True)

        if show_debug and not shown_once: print("Final window data from the baseline period dataset: ", '\n', window_data, '\n')
        
        # Now, we calculate the percentile threshold/climatological mean across the time dimension
        if window_data.time.size > 0:
            # We calculate the climatological mean if that is what is desired
            if calculate_means:
                seas_clim_dict[doy] = window_data.mean(dim = 'time', skipna = True).expand_dims(normalized_doy=[doy])
                
                if show_debug and not shown_once:
                    print("---------------------------------------------------------------------------------------------------------")
                    print(f"Part 4: Store the climatological means across all doys in an empty dictionary!")
                    print("---------------------------------------------------------------------------------------------------------\n")

                    print(f"Dictionary updated for doy {doy} with the time-averaged final window dataset in Part 3.\n\nDictionary entry:\n", seas_clim_dict[doy], '\n')
                    shown_once = True
            
            # Otherwise, we calculate the percentile for a doy and rechunk the result
            else:
                doy_to_save = window_data.chunk({'time':-1}).quantile(percentile_used, dim='time', skipna=True).expand_dims(normalized_doy=[doy])
                doy_to_save = doy_to_save.chunk(optimal_chunks)

                if show_debug: 
                    print("---------------------------------------------------------------------------------------------------------")
                    print(f"Part 4: Save the percentile threshold dataset one single doy at a time!")
                    print("---------------------------------------------------------------------------------------------------------\n")
                    
                # Now, we save the percentile threshold dataset to storage
                save_dataset_to_storage(folder_name_arg=folder_name, sub_folder_name_arg=sub_folder_name, baseline_name_arg=baseline_name,
                                        temp_data_var=temp_data_var_name, custom_id_arg=custom_id, show_debug_arg=show_debug, 
                                        single_download_arg=single_download, current_percentile=chosen_percentile, 
                                        current_doy=doy, ds_to_save=doy_to_save)
    
    if (feb29_doy in unique_doys):
        # If we have a dictionary with our climatological means...
        if calculate_means:
            if feb28_doy in seas_clim_dict and mar1_doy in seas_clim_dict:
                feb_28_ds = seas_clim_dict[feb28_doy].squeeze().drop_vars('normalized_doy')
                mar_1_ds = seas_clim_dict[mar1_doy].squeeze().drop_vars('normalized_doy')
                seas_clim_dict[feb29_doy] = 0.5 * (feb_28_ds + mar_1_ds)
                seas_clim_dict[feb29_doy] = seas_clim_dict[feb29_doy].expand_dims(normalized_doy=[feb29_doy])
                if show_debug: print("Interpolated February 29 dataset (doy 60) in the dictionary:\n", seas_clim_dict[feb29_doy], '\n')
            else:
                print(no_feb29_possible_warning)
        
        # Otherwise, we interpolate percentile thresholds using saved percentile datasets
        else:
            # Quick check to ensure the 60th doy is within our chunk interval
            if (start_doy < feb29_doy) and (feb29_doy < end_doy):
                # We check if we have a chunk list with doy 60 among the missing doys
                if missing_list:
                    if feb29_doy not in missing_list:
                        stop_monitoring = True
                        raise ValueError(missing_list_message_finished)

                # We check for the files required for interpolation        
                file_found_feb28, thresh_feb28_ds = find_file_and_return_it(folder_name_input=folder_name, sub_folder_name_input=sub_folder_name, 
                                                                            baseline_name_input=baseline_name, temp_data_var_input=temp_data_var_name, 
                                                                            custom_id_input=custom_id, percentile_input=chosen_percentile,
                                                                            current_doy=feb28_doy, starting_doy=None, ending_doy=None)
                file_found_mar1, thresh_mar1_ds = find_file_and_return_it(folder_name_input=folder_name, sub_folder_name_input=sub_folder_name, 
                                                                          baseline_name_input=baseline_name, temp_data_var_input=temp_data_var_name, 
                                                                          custom_id_input=custom_id, percentile_input=chosen_percentile,
                                                                          current_doy=mar1_doy, starting_doy=None, ending_doy=None)

                # We proceed with interpolation if both files are found
                if file_found_feb28 and file_found_mar1:
                    doy_to_save = 0.5 * (thresh_feb28_ds + thresh_mar1_ds)
                    doy_to_save = doy_to_save.expand_dims(normalized_doy=[feb29_doy])
                    doy_to_save = doy_to_save.chunk(optimal_chunks)

                    # Now, we save the percentile threshold dataset to storage
                    save_dataset_to_storage(folder_name_arg=folder_name, sub_folder_name_arg=sub_folder_name, baseline_name_arg=baseline_name,
                                        temp_data_var=temp_data_var_name, custom_id_arg=custom_id, show_debug_arg=show_debug, 
                                        single_download_arg=single_download, current_percentile=chosen_percentile, 
                                        current_doy=feb29_doy, ds_to_save=doy_to_save)
                
            # If the two required datasets for interpolating Feb 29 are missing, send a notification!
            else: 
                print(no_feb29_possible_warning)
            
    # We proceed with the full climatology dictionary if we are calculating climatological means
    if calculate_means:
        if show_debug: 
            print("---------------------------------------------------------------------------------------------------------")
            print("Part 5: Creating the complete climatology dataset from the dictionary")
            print("---------------------------------------------------------------------------------------------------------\n")

        # We create the correct coordinates from our dictionary for our final dataset
        doy_coords = np.array(sorted(seas_clim_dict.keys())) # array for full year (1 to 366, if leap)
        if show_debug: print("Dictionary Keys of Registered Unique Day of the Year (doy) Climatological Mean Datasets\n", 
                             "(Should include all values from 1 to 366):\n", doy_coords, '\n')

        # We stack the resulting dictionary datasets while maintaining the correct order
        seas_clim_list = [seas_clim_dict[doy] for doy in doy_coords]
        seas_clim_year = xr.concat(seas_clim_list, dim='normalized_doy')
        seas_clim_year = seas_clim_year.assign_coords(normalized_doy=('normalized_doy', doy_coords))
        seas_clim_year = seas_clim_year.chunk(optimal_chunks)
        
        # Additional chunking of the resulting doy mean/percentile dataset to prevent crashes during saving. 
        
        # For small datasets, it's fine to chunk by the total data available
        if doy_batch_size <= max_doy_chunking_val: 
            seas_clim_year = seas_clim_year.chunk({'normalized_doy': doy_batch_size})
        # For larger datasets with means/percentiles for many doys, we find the best, largest value to chunk by (that produces no remainders)
        else: 
            # The best_chunk_val is the default max_doy_chunking_val value.
            # This code may lead to non-perfect chunking if no best (largest) divisor (under the max allowed) is found and the default is used.
            best_chunk_val = max_doy_chunking_val 

            for divisor in range(max_doy_chunking_val, 0, -1): # Iterate from the maximum allowed divisor down to 1 to determine the best chunk size value
                if doy_batch_size % divisor == 0: # The current largest divisor is found to leave no remainder
                    if divisor != 1: # If we find a divisor that is not 1 (the smallest possible value), we set that as the best chunk size to use
                        best_chunk_val = divisor
                    break # End the best chunk-size (divisor) calculator for loop

            # Lastly, apply the best largest chunk size determined
            seas_clim_year = seas_clim_year.chunk({'normalized_doy': best_chunk_val})

        print("\nFinal (Chunked) Dataset:\n", seas_clim_year, '\n')        
            
            
        if show_debug: 
            print("---------------------------------------------------------------------------------------------------------")
            print(f"Part 6: Saving the dataset!")
            print("---------------------------------------------------------------------------------------------------------\n")
            
        # Coordinate values for batch saving (not full climatology dataset saving)
        coord_values = seas_clim_year['normalized_doy'].values

        # We are saving the dataset in batches
        if batch_saving:
            for i in range(0, len(coord_values), doy_batch_size):
                # First, we gather the starting and ending values of the processed chunk
                start_val = coord_values[i]

                # Quick check to see where to begin downloading a batch from...
                if start_val < start_doy:
                    continue

                # Grab the end index and value
                end_idx = min(i + doy_batch_size, len(coord_values))
                end_val = coord_values[end_idx - 1]
                
                # Save the dataset (subsetting occurs in the function)              
                save_dataset_to_storage(folder_name_arg=folder_name, sub_folder_name_arg=sub_folder_name, baseline_name_arg=baseline_name,
                                        temp_data_var=temp_data_var_name, custom_id_arg=custom_id, show_debug_arg=show_debug,
                                        single_download_arg=single_download, start_val_arg=start_val, end_val_arg=end_val, ds_to_save=seas_clim_year)
        
        # We are saving the full dataset
        else:
            save_dataset_to_storage(folder_name_arg=folder_name, sub_folder_name_arg=sub_folder_name, baseline_name_arg=baseline_name,
                                    temp_data_var=temp_data_var_name, custom_id_arg=custom_id, show_debug_arg=show_debug,
                                    
                                    single_download_arg=single_download, start_val_arg=start_doy, 
                                    end_val_arg=end_doy, ds_to_save=seas_clim_year)  
            
        stop_monitoring = True

    stop_monitoring = True # reset the monitoring before the next loop
''' ------------------------------------------------------------------------------------------------------------------------------------------ ''';


## Set up for the for loop to iterate over for different regions

''' Below is a dictionary of chunk configurations for different subsets of a complete region, or different subfolders' data contained
within a parent folder. As a reminder, for best results, chunk values should be determined by dividing a dataset's coordinates' values 
by a small number less than 400 (to avoid memory issues; smaller numbers across the various coordinates are often better in this regard) 
that leaves no remainder. Accordingly, the example chunk configurations below were determined through the use of the data loading script 
in the cell above. You may also further adjust them here too with the show_debug option, if desired. '''

chunk_configs = {
    "Atlantic": {
        #"Central": {'depth': -1, 'latitude': 101, 'longitude': 209},
        #"Right": {'depth': -1, 'latitude': 276, 'longitude': 204},
        "Top": {'depth': -1, 'latitude': 196, 'longitude': 209}
    },
   # "Pacific": {
    #    "Center": {'depth': -1, 'latitude': 218, 'longitude': 257},
     #   "Left": {'depth': -1, 'latitude': 221, 'longitude': 244}
   # },
   #"Mid": {
   #     "All": {'depth': -1, 'latitude': 253, 'longitude': 270},
    #    "Mid": {'depth': -1, 'latitude': 221, 'longitude': 361}
    #}
}


# Regions to process mean/percentile datasets for (taken from the dataset loading code cell); having this here lets you choose 
# what datasets to process easily when many are loaded into memory.
region_dict_list = [
    {"Atlantic": [#"Central", 
                  "Top", 
                 # "Right"
    ]},
  #  {"Pacific": ["Center", 
                 #"Left"
              #  ]},
   # {"Mid": ["Mid",
        #"All",
   # ]}
]


'''    For convenience, a for loop structure is used to process climatological means/percentile thresholds for folders 
in our directory, whose naming structure are implied by the dictionaries in the list above. Dictionary entries can be 
commented with a # at will to process only select subfolders at a time (within the larger regional folders). ''';

# Accessing the parent folders
for folder in region_dict_list:
    
    # Accessing the (parent folder's filename and the) subfolders within
    for folder_filename, sub_folders in folder.items():
        
        # Initializing the optimal_chunking to be used
        optimal_chunking = None
        
        # Accessing the filename(s) of the subfolder(s)
        for sub_folder_filename in sub_folders:
            
            # Grabbing the optimal chunking settings from the example, custom chunk_configs dictionary above
            optimal_chunking = chunk_configs[folder_filename][sub_folder_filename]
            
            # Calling the function to calculate the mean thresholds (percentile=None)
            calculate_perc_or_clim_given_a_percentile(temp_data = globals()[f'{folder_filename}_obs'].thetao, 
                                                    baseline_slice = baseline_period_slice_choice, 
                                                    folder_name = folder_filename, sub_folder_name = sub_folder_filename, 
                                                   baseline_name = baseline_folder_name, temp_data_var_name="thetao",
                                                   custom_id = custom_id_chosen, optimal_chunks = optimal_chunking, window_half_width = 5, 
                                                   minutes_per_memory_update = 15, chosen_percentile = None,
                                                   start_doy = 1, end_doy = 366, doy_batch_size=20, max_doy_chunking_val=61,
                                                   show_debug = True, single_download = False, 
                                                   missing_list = True, finish_at_missing_list_end = True)
            
            # Calling the function to calculate the percentile thresholds (percentile=INTEGER)
            calculate_perc_or_clim_given_a_percentile(temp_data = globals()[f'{folder_filename}_obs'].thetao, 
                                                     baseline_slice = baseline_period_slice_choice, 
                                                    folder_name = folder_filename, sub_folder_name = sub_folder_filename, 
                                                   baseline_name = baseline_folder_name, temp_data_var_name="thetao",
                                                   custom_id = custom_id_chosen, optimal_chunks = optimal_chunking, window_half_width = 5,
                                                   minutes_per_memory_update = 45, chosen_percentile = percentile,
                                                   start_doy = 1, end_doy = 3, doy_batch_size=1, max_doy_chunking_val=61,
                                                   show_debug = True, single_download = False, 
                                                   missing_list = False, finish_at_missing_list_end = True)

print("We have finished saving all desired doy percentile threshold datasets completely!")
stop_monitoring = True

<div style="color:#008B00; padding: 10px; text-align: center; font-family: Georgia, serif; font-weight: bold; white-space: pre;">ø¤º°`°º¤ø,¸,ø¤°º¤ø,¸¸,ø¤º°`°º¤ø,¸  FILE VALIDATION, VERIFICATION, AND ANIMATIONS  °º¤ø,¸¸,ø¤º°`°º¤ø,¸,ø¤°º¤ø,¸¸,ø¤
</div>

In [ ]:
## Function to create an animation that shows the mean and percentile latitude and longitude maps for the full 1 - 366 period. 
def check_processed_datasets_with_an_animation(baseline_name_arg, folder_name_arg, 
                                               sub_folder_name_arg=None,
                                               custom_output_filename=None, 
                                               percentile=None, chosen_depth=0.494):
    # First determine if the animation is that of a climatological means/percentile thresholds dataset
    if percentile == None:
        show_climatological_means = True 
    else:
        show_climatological_means = False
    
    # Use a subfolder name in the filepath if a subfolder name is used!
    if sub_folder_name_arg == None or sub_folder_name_arg == "":
        accessing_subfolder = False
    else:
        accessing_subfolder = True
        
    # Reset the other parameters for later code (if making a means animation)
    if show_climatological_means:
        percentile = 0
        
    ## Gather the stored dataset filepaths
    if show_climatological_means:
        data_type = "Clim"
        my_root_directory = clim_data_directory
    else:
        data_type = f"Thresh{percentile}th"
        my_root_directory = perc_data_directory
        
    if accessing_subfolder: # Accessing a subfolder in a parent folder
        full_id   = f"{folder_name_arg}_{sub_folder_name_arg}" # Ex. "Atlantic_Top"
        data_path = f"{baseline_folder_name}/{folder_name_arg}/{full_id}"
    else: # Only accessing the parent folder!
        full_id = folder_name_arg
        data_path = f"{baseline_folder_name}/{folder_name_arg}"
        
    # Determine the final path and grab its files (which share the "{full_id}_*{baseline_name_arg}.zarr" naming convention!
    final_file_path = f'{my_root_directory}/{data_path}/{full_id}_*{baseline_name_arg}.zarr'
    paths = glob.glob(final_file_path)
    
    # A quick check to ensure we have located files given our arguments
    if not paths:
        start_error = "No files found matching the pattern"
        cont_error = "\nPlease verify you inputted the proper baseline_name, folder_name, sub_folder_name, percentile, and show_climatological_means arguments!"
        raise FileNotFoundError(f"{start_error}:\n{final_file_path}/{full_id}_thetao...300m...{baseline_name_arg}.zarr\n{cont_error}")
    
    
    ## Fill a dictionary where all (1 to 366) doys are matched with their corresponding filepaths
    doys_dict = {}
    
    for filepath in paths:
        # Open and check what doys are in this file
        ds = xr.open_zarr(filepath)
        
        # In my earlier percentile datasets (calculated in the same manner), my normalized_doys were saved as variable doys instead.
        # this following if statement will likely be unnecessary for you.
        if 'doy' in ds.coords:
            ds = ds.rename({'doy': 'normalized_doy'}).expand_dims('normalized_doy')
        
        ds = ds.thetao
        file_doys = ds['normalized_doy'].values

        # Handle both single value and arrays
        if np.isscalar(file_doys):
            file_doys = [file_doys]
         
        # Map each doy to its file
        for doy in file_doys:
            doys_dict[int(doy)] = filepath
         
        ds.close()
    
    
    ## Use a file and its features to set up the plot
    available_doys = sorted(doys_dict.keys())
    setup_file = doys_dict[available_doys[0]]
    
    if not show_climatological_means:
        setup_ds = xr.open_zarr(filepath).thetao.drop_vars("quantile")
    else:
        setup_ds = xr.open_zarr(filepath).thetao
    
    # Quick fix for my personal, early datasets
    if 'doy' in setup_ds.coords:
        setup_ds = setup_ds.rename({'doy': 'normalized_doy'}).expand_dims('normalized_doy')
    
    # Depth selection
    setup_ds = setup_ds.sel(depth=chosen_depth, method='nearest')
    first_depth = round(setup_ds.depth.item(), 1)
    setup_ds = setup_ds.drop_vars("depth")

    lon = setup_ds.longitude.values
    lat = setup_ds.latitude.values
    
    # Check for single or multiple-doys in the setup dataset, and return the thetao data for just one (the first) doy
    if len(setup_ds['normalized_doy'].values.shape) == 0 or setup_ds['normalized_doy'].values.size == 1:
        # Single day file
        setup_ds   = setup_ds.drop_vars("normalized_doy").squeeze()
        setup_data = setup_ds.values
    else:
        # Multi-day file
        setup_data = setup_ds.isel(normalized_doy=0).values
    
    setup_ds.close()
    
    
    ## Initialize the plot    
    fig, ax = plt.subplots(figsize=(14, 6), 
                           subplot_kw={'projection': ccrs.Mercator()})
    
    pcm = ax.pcolormesh(
        lon, lat, setup_data,
        cmap='RdYlBu_r',
        vmin=-5, vmax=35,
        transform=ccrs.PlateCarree(),
    )
    
    ax.set_extent([0, 360, -30, 90], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, color='lightgray')
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    
    title = ax.set_title('')
    title_base = '(Relative to 1993-2022)' if baseline_name_arg == 'Baseline9322' else f'({baseline_name_arg})'
    
    
    ## Animation function
    def animate(i):
        doy = available_doys[i]
        filepath = doys_dict[doy]
        
        # Load the dataset
        ds = xr.open_zarr(filepath).thetao.sel(depth=chosen_depth, method='nearest').drop_vars("depth")
        
        # Quick fix for my personal, early datasets
        if 'doy' in ds.coords:
            ds = ds.rename({'doy': 'normalized_doy'}).expand_dims('normalized_doy')
        
        if not show_climatological_means:
            ds = ds.drop_vars("quantile")
            
        # Check if this is a single-day or multi-day file
        doy_values = ds['normalized_doy'].values
        
        # Load the data if available for a single doy or select the correct doy in a dataset
        if np.isscalar(doy_values) or doy_values.size == 1:
            frame_data = ds.values
        else:
            doy_idx = np.where(doy_values == doy)[0][0]
            frame_data = ds.isel(normalized_doy=doy_idx).values
        
        # Update the plot
        pcm.set_array(frame_data.ravel())
        title.set_text(f'{data_type} Day {doy} of the Year\n{title_base}')
        
        ds.close()
        return pcm, title
    
    
    ## Create the resulting animation
    type_message = "climatological means" if show_climatological_means else "percentile thresholds"
    print(f"Began animation for the {type_message} of the {data_type} datasets!")
    
    chosen_doys = len(available_doys)
    
    anim = animation.FuncAnimation(
        fig, animate,
        frames=chosen_doys,
        interval=200,
        blit=False,
        repeat=True
    )
    
    writer = animation.PillowWriter(fps=2)
    
    if custom_output_filename == None:
        output_filename = f"{full_id}_{data_type}_{baseline_name_arg}_{chosen_doys}_doys_total_with_depth_of_{first_depth}.gif"
    else:
        output_filename = custom_output_filename
    print(f"Saving animation at: {output_filename}") 
    
    
    ## Save the resulting animation
    def print_frame_progress(current_frame, total_frames):
        print(f"\r → Doy (Frame) Processed: {current_frame + 1}/{total_frames}", end='', flush=True)

    #with ProgressBar():
    anim.save(output_filename, writer=writer, dpi=100,
              progress_callback=print_frame_progress)
    
    plt.tight_layout()
    plt.close(fig)
    print(f"\nAnimation finished and saved!\n")
    
    return anim

# ------------------------------------------------------------------------------------------------------------------------------


## Examples for creating the animations (with a loop)

# Include all the subfolders in a list like this, to call upon and make animations for each unique subregion
# Or you could provide a sub_folder_name_arg argument in the function(s) below manually
perc_anims_to_make = ["Top"] 

for sub_folder in perc_anims_to_make:    
    check_processed_datasets_with_an_animation(baseline_name_arg="Baseline9322", folder_name_arg="Atlantic", 
                                               sub_folder_name_arg=sub_folder,
                                               percentile=90, chosen_depth=300)
     
check_processed_datasets_with_an_animation(baseline_name_arg="Baseline9322", folder_name_arg="Atlantic", 
                                           sub_folder_name_arg="Full",
                                           percentile=None, chosen_depth=300)